In [ ]:
!pip install "torch==2.10.0" "pandas==2.2.2" xgrammar
!pip install accelerate pillow
!pip install --upgrade transformers accelerate fastapi uvicorn pyngrok -q
!pip install -U bitsandbytes

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import kagglehub
import gc
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
import xgrammar as xgr

# 1. Limpieza rigurosa
gc.collect()
torch.cuda.empty_cache()

print("Descargando/Verificando modelo...")
MODEL_PATH = kagglehub.model_download("google/gemma-4/transformers/gemma-4-e4b-it")

print("Cargando procesador...")
processor = AutoProcessor.from_pretrained(MODEL_PATH)

print("Cargando modelo en Transformers...")

# CAMBIO CRÍTICO: Eliminamos la clave "cpu" y subimos un poco el límite de las T4.
# Las T4 tienen ~15GB, le damos 13GB a cada una para que entre todo el modelo
# y nos deje ~2GB libres por tarjeta para el contexto y FastAPI.
max_memory = {0: "13GiB", 1: "13GiB"} 

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory, # Forzamos a que todo se quede en las GPUs
    dtype=torch.float16,
)

print("Inicializando XGrammar...")
tokenizer_info = xgr.TokenizerInfo.from_huggingface(processor.tokenizer)
print("¡Modelo cargado exitosamente en las GPUs!")

Descargando/Verificando modelo...
Cargando procesador...
Cargando modelo en Transformers...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Inicializando XGrammar...
¡Modelo cargado exitosamente en las GPUs!


In [ ]:
import os
import threading
import uvicorn
import json
import torch
import hashlib
import traceback
import asyncio
import uuid
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import LogitsProcessorList
import xgrammar as xgr

app = FastAPI()

# Almacén de jobs en memoria
jobs = {}
compiled_grammars_cache = {}

class GenerateRequest(BaseModel):
    prompt: str = ""
    max_tokens: int = 512
    schema_dict: dict = None

@app.post("/generate")
async def generate(req: GenerateRequest):
    """Retorna inmediatamente un job_id. La inferencia corre en background."""
    job_id = str(uuid.uuid4())
    jobs[job_id] = {"status": "pending", "response": None, "error": None}
    asyncio.create_task(run_inference(job_id, req))
    return {"job_id": job_id}

@app.get("/status/{job_id}")
async def get_status(job_id: str):
    if job_id not in jobs:
        return {"error": "job_id not found"}
    return jobs[job_id]

@app.get("/health")
async def health():
    return {"status": "ok"}

async def run_inference(job_id: str, req: GenerateRequest):
    """Tarea en background: hace la inferencia y guarda el resultado."""
    try:
        loop = asyncio.get_event_loop()
        result = await loop.run_in_executor(None, _blocking_inference, req)
        jobs[job_id] = {"status": "done", "response": result, "error": None}
        print(f"✅ Job {job_id} completado.")
    except Exception as e:
        jobs[job_id] = {"status": "error", "response": None, "error": str(e)}
        print(f"❌ Job {job_id} falló: {e}")
        print(traceback.format_exc())

def _blocking_inference(req: GenerateRequest) -> str:
    """La inferencia real, bloqueante, corre en thread separado."""
    global compiled_grammars_cache, processor, model, tokenizer_info

    inputs = processor(text=req.prompt, return_tensors="pt").to(model.device)
    logits_processor_list = None

    if req.schema_dict:
        schema_str = json.dumps(req.schema_dict)
        schema_hash = hashlib.md5(schema_str.encode()).hexdigest()

        if schema_hash not in compiled_grammars_cache:
            print("⚙️ Compilando XGrammar por primera vez para este esquema...")
            compiler = xgr.GrammarCompiler(tokenizer_info)
            compiled_grammars_cache[schema_hash] = compiler.compile_json_schema(schema_str)
            print("✅ XGrammar compilado y guardado en caché.")
        else:
            print("⚡ Esquema encontrado en caché. Saltando compilación.")

        compiled_grammar = compiled_grammars_cache[schema_hash]
        xgr_processor = xgr.contrib.hf.LogitsProcessor(compiled_grammar)
        logits_processor_list = LogitsProcessorList([xgr_processor])

    print("🔄 Iniciando generación en la GPU...")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            logits_processor=logits_processor_list if logits_processor_list else None,
            pad_token_id=processor.tokenizer.eos_token_id
        )

    prompt_length = inputs["input_ids"].shape[1]
    text = processor.decode(outputs[0][prompt_length:], skip_special_tokens=True)

    del inputs, outputs
    torch.cuda.empty_cache()

    print("📤 Respuesta generada:")
    print(text)
    return text

# --- Exponer con ngrok ---
ngrok.set_auth_token("YOUR_API_KEY_HERE")
public_url = ngrok.connect(8000)
print(f"\n🚀 URL pública: {public_url}\n")

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

import time
while True:
    time.sleep(60)

In [ ]:
import os
import signal
import subprocess

try:
    # Find the process ID (PID) using port 8000
    pid = int(subprocess.check_output(["lsof", "-t", "-i:8000"]).strip())
    # Kill the process
    os.kill(pid, signal.SIGKILL)
    print(print(f"✅ Successfully killed process {pid} on port 8000."))
except subprocess.CalledProcessError:
    print("ℹ️ No process was found running on port 8000. It is already free!")
